# 02: Data Preprocessing & Semantic-Group Split

Normalizes text, encodes multi-task labels, and performs leak-free 70:15:15 semantic group splitting.

In [ ]:
import pandas as pd
from src.data.loader import load_or_create_dataset
from src.data.preprocessing import MultiTaskLabelEncoder, group_aware_split, normalize_text
from src.utils.config import PROCESSED_DATA_DIR

df = load_or_create_dataset()
df['text'] = df['text'].apply(normalize_text)

# Encode multi-head categorical targets
encoder = MultiTaskLabelEncoder()
encoded_targets = encoder.encode_dataframe(df)
for head, arr in encoded_targets.items():
    df[f"{head}_encoded"] = arr

print("Encoded targets sample:")
df[['id', 'text', 'risk_level_encoded', 'intent_encoded']].head()

In [ ]:
# Perform 70:15:15 Semantic-Group-Aware Split
train_df, val_df, test_df = group_aware_split(df, group_col='semantic_group', train_ratio=0.7, val_ratio=0.15, test_ratio=0.15)
print(f"Train samples: {len(train_df)}, Val samples: {len(val_df)}, Test samples: {len(test_df)}")

# Verify zero semantic leakage
train_groups = set(train_df['semantic_group'])
test_groups = set(test_df['semantic_group'])
assert train_groups.isdisjoint(test_groups), "Leakage detected between Train and Test!"
print("Zero semantic leakage verified across splits.")

# Save splits
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
train_df.to_csv(PROCESSED_DATA_DIR / 'train.csv', index=False)
val_df.to_csv(PROCESSED_DATA_DIR / 'val.csv', index=False)
test_df.to_csv(PROCESSED_DATA_DIR / 'test.csv', index=False)
print("Processed splits successfully saved.")